In [1]:
import pandas as pd
import numpy as np
import json
import gzip
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy

sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv('../data/processed/interactions_filtered.csv')
print(f"Interactions: {df.shape}")
df.head()

Interactions: (40879, 10)


,user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
0,8842281e1d1347389f2ab93d60773d4d,23310161,f4b4b050f4be00e9283c92a814af2670,True,4,Fun sequel to the original.,Tue Nov 17 11:37:35 -0800 2015,Tue Nov 17 11:38:05 -0800 2015,NaN,NaN
1,8842281e1d1347389f2ab93d60773d4d,817720,75fd46041466ceb406b7fd69b089b9c5,True,5,NaN,Wed May 20 21:29:23 -0700 2015,Wed May 20 21:29:23 -0700 2015,NaN,NaN
2,8842281e1d1347389f2ab93d60773d4d,1969280,5809d5592ee32745e048a9c67ac27100,True,5,NaN,Sat Nov 08 08:56:58 -0800 2014,Wed Dec 17 00:37:25 -0800 2014,NaN,NaN
3,8842281e1d1347389f2ab93d60773d4d,17290220,22d424a2b0057b18fb6ecf017af7be92,True,5,One of my favorite books to read to my 5 year ...,Sat Nov 08 08:54:03 -0800 2014,Wed Jan 25 13:56:12 -0800 2017,Tue Jan 24 00:00:00 -0800 2017,NaN
4,8842281e1d1347389f2ab93d60773d4d,1027760,0c8a75acde799d70696f4aecf2d611de,True,4,NaN,Tue Mar 18 22:23:03 -0700 2014,Wed Mar 22 11:47:31 -0700 2017,NaN,NaN


In [3]:
books = []
with gzip.open('../data/raw/goodreads_books_children.json.gz', 'rt') as f:
    for line in f:
        books.append(json.loads(line))

df_books = pd.DataFrame(books)
print(f"Books loaded: {df_books.shape}")
print(f"Columns: {df_books.columns.tolist()}")
df_books.head(3)

Books loaded: (124082, 29)
Columns: ['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series']


,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,1599150603,7,[],US,,"[{'count': '56', 'name': 'to-read'}, {'count':...",,false,4.13,B00DU10PUG,...,9,,2006,https://www.goodreads.com/book/show/287141.The...,https://s.gr-assets.com/assets/nophoto/book/11...,287141,46,278578,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls
1,1934876569,6,[151854],US,,"[{'count': '515', 'name': 'to-read'}, {'count'...",,false,4.22,,...,3,,2009,https://www.goodreads.com/book/show/6066812-al...,https://images.gr-assets.com/books/1316637798m...,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...
2,0590417010,193,[],US,eng,"[{'count': '450', 'name': 'to-read'}, {'count'...",,false,4.43,B017RORXNI,...,9,,1995,https://www.goodreads.com/book/show/89378.Dog_...,https://images.gr-assets.com/books/1360057676m...,89378,1331,86259,Dog Heaven,Dog Heaven


In [4]:
# See all columns
print(df_books.columns.tolist())

# Keep only the useful columns for recommendations
cols_to_keep = ['book_id', 'title', 'authors', 'average_rating', 
                'popular_shelves', 'description', 'publication_year']

# Check which of these actually exist
available = [c for c in cols_to_keep if c in df_books.columns]
missing = [c for c in cols_to_keep if c not in df_books.columns]
print(f"\nAvailable: {available}")
print(f"Missing:   {missing}")

df_books_slim = df_books[available].copy()
df_books_slim.head(3)

['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series']

Available: ['book_id', 'title', 'authors', 'average_rating', 'popular_shelves', 'description', 'publication_year']
Missing:   []


,book_id,title,authors,average_rating,popular_shelves,description,publication_year
0,287141,The Aeneid for Boys and Girls,"[{'author_id': '3041852', 'role': ''}]",4.13,"[{'count': '56', 'name': 'to-read'}, {'count':...","Relates in vigorous prose the tale of Aeneas, ...",2006
1,6066812,All's Fairy in Love and War (Avalon: Web of Ma...,"[{'author_id': '19158', 'role': ''}]",4.22,"[{'count': '515', 'name': 'to-read'}, {'count'...","To Kara's astonishment, she discovers that a p...",2009
2,89378,Dog Heaven,"[{'author_id': '5411', 'role': ''}]",4.43,"[{'count': '450', 'name': 'to-read'}, {'count'...",In Newbery Medalist Cynthia Rylant's classic b...,1995


In [5]:
print("Interactions book_id sample:")
print(df['book_id'].head())
print(f"dtype: {df['book_id'].dtype}")

print("\nBooks book_id sample:")
print(df_books_slim['book_id'].head())
print(f"dtype: {df_books_slim['book_id'].dtype}")

Interactions book_id sample:
0    23310161
1      817720
2     1969280
3    17290220
4     1027760
Name: book_id, dtype: int64
dtype: int64

Books book_id sample:
0     287141
1    6066812
2      89378
3    3209312
4    1698376
Name: book_id, dtype: str
dtype: str


In [6]:
# Align dtypes so the join works correctly
df['book_id'] = df['book_id'].astype(str)
df_books_slim['book_id'] = df_books_slim['book_id'].astype(str)

# Join books metadata onto interactions
df_merged = df.merge(df_books_slim, on='book_id', how='left')

print(f"Interactions before merge: {len(df):,}")
print(f"Interactions after merge:  {len(df_merged):,}")
print(f"Books matched:   {df_merged['title'].notna().sum():,}")
print(f"Books unmatched: {df_merged['title'].isna().sum():,}")

df_merged[['user_id', 'book_id', 'rating', 'title', 'authors', 'average_rating']].head()

Interactions before merge: 40,879
Interactions after merge:  40,879
Books matched:   40,879
Books unmatched: 0


,user_id,book_id,rating,title,authors,average_rating
0,8842281e1d1347389f2ab93d60773d4d,23310161,4,The Day the Crayons Came Home,"[{'author_id': '6561846', 'role': ''}, {'autho...",4.43
1,8842281e1d1347389f2ab93d60773d4d,817720,5,Babar and His Children,"[{'author_id': '4788791', 'role': ''}, {'autho...",3.92
2,8842281e1d1347389f2ab93d60773d4d,1969280,5,"Iggy Peck, Architect","[{'author_id': '379125', 'role': ''}, {'author...",4.44
3,8842281e1d1347389f2ab93d60773d4d,17290220,5,"Rosie Revere, Engineer","[{'author_id': '379125', 'role': ''}, {'author...",4.54
4,8842281e1d1347389f2ab93d60773d4d,1027760,4,Madeline and the Bad Hat,"[{'author_id': '64280', 'role': ''}]",4.27


In [7]:
# Drop authors column since it only has IDs not names
# We'll keep title, average_rating, publication_year
df_merged = df_merged[['user_id', 'book_id', 'rating', 'title', 
                        'average_rating', 'publication_year']].copy()

# Save it
df_merged.to_csv('../data/processed/interactions_with_metadata.csv', index=False)
print("Saved to data/processed/interactions_with_metadata.csv")
df_merged.head()

Saved to data/processed/interactions_with_metadata.csv


,user_id,book_id,rating,title,average_rating,publication_year
0,8842281e1d1347389f2ab93d60773d4d,23310161,4,The Day the Crayons Came Home,4.43,2015
1,8842281e1d1347389f2ab93d60773d4d,817720,5,Babar and His Children,3.92,1966
2,8842281e1d1347389f2ab93d60773d4d,1969280,5,"Iggy Peck, Architect",4.44,2007
3,8842281e1d1347389f2ab93d60773d4d,17290220,5,"Rosie Revere, Engineer",4.54,2013
4,8842281e1d1347389f2ab93d60773d4d,1027760,4,Madeline and the Bad Hat,4.27,1957


In [8]:
# Rebuild surprise dataset with the merged data
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_merged[['user_id', 'book_id', 'rating']], reader)

# Train on full dataset this time (no test split — we already evaluated)
trainset = data.build_full_trainset()
model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
model.fit(trainset)

print("Model trained on full dataset")

Model trained on full dataset


In [9]:
# Book ID to title lookup
book_titles = df_merged.set_index('book_id')['title'].to_dict()
book_ratings = df_merged.groupby('book_id')['rating'].count()
reliable_books = book_ratings[book_ratings >= 10].index.astype(str)

def get_recommendations_named(user_id, n=10):
    rated_books = set(df_merged[df_merged['user_id'] == user_id]['book_id'].values)
    candidate_books = [b for b in reliable_books if b not in rated_books]
    
    predictions = [model.predict(user_id, book_id) for book_id in candidate_books]
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    print(f"Top {n} recommendations for user {user_id[:8]}...:\n")
    for i, pred in enumerate(predictions[:n], 1):
        title = book_titles.get(pred.iid, "Unknown")
        print(f"{i:2}. {title:<50} predicted: {pred.est:.2f}")

# Try it on a few users
for user_id in df_merged['user_id'].unique()[2:5]:
    user_avg = df_merged[df_merged['user_id'] == user_id]['rating'].mean()
    print(f"\nUser {user_id[:8]}... | avg rating: {user_avg:.2f}")
    get_recommendations_named(user_id, n=5)
    print("-" * 70)


User 06316bec... | avg rating: 4.57
Top 5 recommendations for user 06316bec...:

 1. It's a Book                                        predicted: 4.91
 2. Wonder (Wonder #1)                                 predicted: 4.83
 3. Waiting                                            predicted: 4.78
 4. The Night Gardener                                 predicted: 4.78
 5. Mike Mulligan and His Steam Shovel                 predicted: 4.75
----------------------------------------------------------------------

User 1711b2a4... | avg rating: 3.50
Top 5 recommendations for user 1711b2a4...:

 1. Press Here                                         predicted: 4.35
 2. It's a Book                                        predicted: 4.31
 3. George                                             predicted: 4.31
 4. A Light in the Attic                               predicted: 4.31
 5. The Fantastic Flying Books of Mr. Morris Lessmore  predicted: 4.31
-------------------------------------------------------

In [10]:
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs':  [20, 30],
    'lr_all':    [0.005, 0.010],
    'reg_all':   [0.02, 0.05]
}

gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse'],
    cv=5,
    n_jobs=-1  # use all CPU cores
)

gs.fit(data)

print(f"Best RMSE: {gs.best_score['rmse']:.4f}")
print(f"Best params: {gs.best_params['rmse']}")
print(f"\nBaseline RMSE:       0.9060")
print(f"Default SVD RMSE:    0.7917")
print(f"Tuned SVD RMSE:      {gs.best_score['rmse']:.4f}")

Best RMSE: 0.7850
Best params: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05}

Baseline RMSE:       0.9060
Default SVD RMSE:    0.7917
Tuned SVD RMSE:      0.7850


In [12]:
import pickle
import json

# Re-train the best model properly on the full dataset
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_merged[['user_id', 'book_id', 'rating']], reader)
trainset = data.build_full_trainset()

# Use the best params from grid search
best_params = gs.best_params['rmse']
print(f"Training with best params: {best_params}")

final_model = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)
final_model.fit(trainset)
print("Model trained!")

# Save everything
with open('../app/model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

book_titles = df_merged.set_index('book_id')['title'].to_dict()
with open('../app/book_titles.json', 'w') as f:
    json.dump(book_titles, f)

book_ratings_count = df_merged.groupby('book_id')['rating'].count()
reliable = book_ratings_count[book_ratings_count >= 10].index.astype(str).tolist()
with open('../app/reliable_books.json', 'w') as f:
    json.dump(reliable, f)

users = df_merged['user_id'].unique().tolist()
with open('../app/users.json', 'w') as f:
    json.dump(users, f)

print("All files saved to app/")

Training with best params: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05}
Model trained!
All files saved to app/
